# Welles — Colab GPU API

Runs the same JSON API Vercel expects (`POST {mode, prompt, maxNewTokens}` → `{text}`).

**Runtime → Change runtime type → T4 GPU**

Keep this tab open while you use the site. When Colab disconnects, the URL dies — run again and update `WELLES_API_URL`.

In [ ]:
%pip install -q fastapi uvicorn peft transformers accelerate bitsandbytes nest_asyncio pyngrok

## Optional: ngrok (stable public URL)
Free account → https://dashboard.ngrok.com/get-started/your-authtoken  
Paste the token when asked. If you skip this, the notebook prints only a local address (not reachable from Vercel).

In [ ]:
from getpass import getpass
import os

token = getpass("ngrok authtoken (Enter to skip): ").strip()
if token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = token
    os.environ["WELLES_USE_NGROK"] = "1"
    print("ngrok token set")
else:
    os.environ["WELLES_USE_NGROK"] = "0"
    print("No ngrok — Vercel will not reach this runtime until you add a tunnel.")

In [ ]:
import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from peft import PeftModel
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer
import threading

nest_asyncio.apply()

ADAPTER_ID = "n0social/welles"
BASE_ID = "Qwen/Qwen3-8B"
SYSTEM = (
    "You are Welles, a writer in the voice of Orson Welles: "
    "oratorical, cinematic, and deliberate."
)
MODES = {
    "Write": "Write the following in your voice.",
    "Rewrite": "Rewrite the following in your voice. Keep the substance.",
    "Continue": "Continue from the following in your voice. Hold one argument to the end.",
}

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    torch_dtype=torch.float16,
    device_map={ "": 0 },
)
model = PeftModel.from_pretrained(model, ADAPTER_ID)
model.eval()
print("model ready")

class Body(BaseModel):
    mode: str = "Write"
    prompt: str = ""
    maxNewTokens: int = Field(default=768, alias="maxNewTokens")
    max_new_tokens: int | None = None

    class Config:
        populate_by_name = True

api = FastAPI()
api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@api.get("/health")
def health():
    return {"ok": True}

@api.post("/")
@api.post("/generate")
def generate(body: Body):
    text = (body.prompt or "").strip()
    if not text:
        return {"text": "Enter a brief or draft first."}
    mode = body.mode if body.mode in MODES else "Write"
    max_new = body.max_new_tokens or body.maxNewTokens or 768
    max_new = max(256, min(2048, int(max_new)))

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{MODES[mode]}\n\n{text}"},
    ]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(rendered, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[-1] :]
    return {"text": tokenizer.decode(new_tokens, skip_special_tokens=True).strip()}

PORT = 7860

def run():
    uvicorn.run(api, host="0.0.0.0", port=PORT, log_level="info")

threading.Thread(target=run, daemon=True).start()
print("API listening on", PORT)

public = None
if os.environ.get("WELLES_USE_NGROK") == "1":
    from pyngrok import ngrok
    public = ngrok.connect(PORT, "http").public_url
    print("\nWELLES_API_URL=", public)
    print("Paste that into Vercel env WELLES_API_URL (and .env.local for local Next.js).")
else:
    print("Set an ngrok token in the previous cell so Vercel can reach this API.")

Leave this runtime running. Test with:

```python
import requests
r = requests.post(public or "http://127.0.0.1:7860/", json={
    "mode": "Write",
    "prompt": "A short paragraph on the camera as a witness.",
    "maxNewTokens": 256,
})
print(r.status_code, r.json())
```